# Importing Libraries

In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG      = "workspace"
BRONZE_TABLE = f"{CATALOG}.sme_risk_bronze.asic_companies_raw"
SILVER_TABLE = f"{CATALOG}.sme_risk_silver.companies"

RENAME_MAP = {
    "company_name":                   "company_name",
    "acn":                            "acn",
    "type":                           "company_type",
    "class":                          "company_class",
    "sub_class":                      "company_sub_class",
    "status":                         "status_code",
    "date_of_registration":           "registration_date",
    "date_of_deregistration":         "deregistration_date",
    "previous_state_of_registration": "previous_state",
    "state_registration_number":      "state_registration_number",
    "modified_since_last_report":     "modified_since_last_report",
    "current_name_indicator":         "current_name_indicator",
    "abn":                            "abn",
    "current_name":                   "current_name",
    "current_name_start_date":        "current_name_start_date",
}

# Code labels: ASIC Company Dataset Help File, Appendix Item 2
TYPE_LABELS = {
    "APTY": "Australian proprietary company", "APUB": "Australian public company",
    "FNOS": "Foreign company registered in Australia", "RACN": "Registered Australian corporation",
    "CCIV": "Corporate collective investment vehicle (not in ASIC help file)",
}
CLASS_LABELS = {
    "LMSH": "Limited by shares", "LMGT": "Limited by guarantee", "LMSG": "Limited by shares and guarantee",
    "NLIA": "No liability", "UNLM": "Unlimited", "NONE": "No Australian equivalent",
}
SUB_CLASS_LABELS = {
    "EXPT": "Exempt proprietary", "NEXT": "Non-exempt proprietary", "PROP": "Proprietary other",
    "PNPC": "Proprietary non-profit", "PSTC": "Proprietary superannuation trustee", "HUNT": "Proprietary home unit",
    "LIST": "Listed public", "ULST": "Unlisted public", "ULSN": "Unlisted public non-profit",
    "ULSS": "Unlisted public superannuation trustee", "NLTD": "Non-profit public without 'Limited'",
    "LISN": "Licensed to omit 'Limited'", "LISS": "Licensed to omit 'Limited' (super trustee)",
    "PUBF": "Foreign company lodging balance sheet", "RACA": "Registrable Australian corporation (association)",
    "RACO": "Registrable Australian corporation (non-association)",
    "STFI": "Public small transferring financial institution", "NONE": "Unknown",
    "WHSL":"Wholesale CCIV (not in ASIC help file; inferred)",
}
STATUS_LABELS = {
    "REGD": "Registered", "DRGD": "Deregistered", "EXAD": "External administration",
    "SOFF": "Strike-off action in progress", "NOAC": "Not active", "NRGD": "Not registered",
    "PROV": "Provisional", "DISS": "Dissolved by Act of Parliament", "DIV3": "Transferred via Div 3",
    "PEND": "Pending (schemes)", "CNCL": "Cancelled / transferred to another law",
}

PREV_STATE_LABELS = {
    "NSW": "New South Wales", "VIC": "Victoria", "QLD": "Queensland",
    "WA": "Western Australia", "SA": "South Australia", "TAS": "Tasmania",
    "ACT": "Australian Capital Territory", "NT": "Northern Territory",
    "NFK": "Norfolk Island",
}

# Reading From the Bronze Table

**Select the latest snapshot.** Bronze keeps every weekly ASIC file (one full copy of the register per `_snapshot_date`).
Silver describes companies as they are now, so only the most recent snapshot is used; without this filter every company would appear once per week loaded.


In [0]:
bronze = spark.table(BRONZE_TABLE)
# Bronze keeps every weekly snapshot; Silver is rebuilt from the latest one.
snapshot_date = bronze.agg(F.max("_snapshot_date")).first()[0]

df = (bronze
      .filter(F.col("_snapshot_date") == F.lit(snapshot_date))
      .drop("_source_file", "_source_url", "_ingested_at"))

baseline_rows = df.count()
baseline_acns = df.select("acn").distinct().count()
print(f"Snapshot {snapshot_date}: {baseline_rows:,} rows, {baseline_acns:,} distinct ACNs")

# Data Transformations

In [0]:
for old, new in RENAME_MAP.items():
    df = df.withColumnRenamed(old, new)

### Transform: `company_name`
Decisions are based on profiling of the 2026-10-12 snapshot and checks against ASIC's online register:
- **CSV escaping decoded:** 429 names were wrapped in quotes with inner quotes doubled
  (e.g. `"GO ""N"" OFF PTY LTD"` → `GO "N" OFF PTY LTD`, verified on ASIC for ACNs 106 366 090 and 101 413 405).
- **Formatting cleaned:** 45,798 names with leading/trailing spaces and 46,776 with double spaces.
- **Kept unchanged:** asterisks (genuine, e.g. `CREATE*IT PTY LTD`), numbers in names, case, punctuation.
- **Flagged, not changed:** º (U+00BA), which appears to replace | in ASIC's export (PILOTPATH example); pending further evidence.
- **New feature:** `is_acn_named`, true when the company's own ACN appears in its name (e.g. `ACN 094 072 214 PTY LTD`, or an ABN-style name containing the ACN).

In [0]:
def unescape_csv(col_name):
    """Remove CSV outer quotes and turn doubled quotes back into single ones."""
    c = F.col(col_name)
    return (F.when(c.startswith('"') & c.endswith('"') & (F.length(c) >= 2),
                   F.regexp_replace(F.expr(f"substring({col_name}, 2, length({col_name}) - 2)"), '""', '"'))
             .otherwise(c))

def clean_text(col_name):
    """Strip control characters, trim, collapse repeated spaces, blank → null."""
    c = F.regexp_replace(F.col(col_name), r"[\x00-\x1F\x7F]", "")
    c = F.regexp_replace(F.trim(c), r"\s{2,}", " ")
    return F.when(c == "", None).otherwise(c)

# 1) Undo CSV escaping on the two name columns
df = (df.withColumn("company_name", unescape_csv("company_name"))
        .withColumn("current_name", unescape_csv("current_name")))

# 2) Clean every string column (fixes name spaces, previous_state trailing spaces, etc.)
string_cols = {f.name for f in df.schema.fields if f.dataType.simpleString() == "string"}
df = df.select([clean_text(c).alias(c) if c in string_cols else F.col(c) for c in df.columns])

# 3) Name-based flags
df = (df
    .withColumn("company_name_has_export_artifact",
                F.coalesce(F.col("company_name").rlike("[\u00BA\u00B0]"), F.lit(False)))   # º (U+00BA) as in the file; ° kept just in case
    .withColumn("is_acn_named", F.coalesce(
        F.instr(F.regexp_replace(F.col("company_name"), r"[^0-9]", ""), F.col("acn")) > 0, F.lit(False))))

### Parse dates, ABN and flags
Dates DD/MM/YYYY → DATE (unparseable flags computed first), ABN `0` → null, `modified_since_last_report` Y → true / blank → false.

In [0]:
def parse_ddmmyyyy(col_name):
    return F.expr(f"try_to_timestamp({col_name}, 'dd/MM/yyyy')").cast("date")

df = (df
    # flags first, while the raw text is still there
    .withColumn("registration_date_unparseable",
                F.col("registration_date").isNotNull() & parse_ddmmyyyy("registration_date").isNull())
    .withColumn("deregistration_date_unparseable",
                F.col("deregistration_date").isNotNull() & parse_ddmmyyyy("deregistration_date").isNull())
    .withColumn("current_name_start_date_unparseable",
                F.col("current_name_start_date").isNotNull() & parse_ddmmyyyy("current_name_start_date").isNull())
    # dates
    .withColumn("registration_date", parse_ddmmyyyy("registration_date"))
    .withColumn("deregistration_date_reported", parse_ddmmyyyy("deregistration_date"))
    .withColumn("current_name_start_date", parse_ddmmyyyy("current_name_start_date"))
    .drop("deregistration_date")
    # ABN: "0" means no ABN
    .withColumn("abn", F.when(F.col("abn") == "0", None).otherwise(F.col("abn")))
    # Y → true, blank → false (plain == 'Y' would give null)
    .withColumn("modified_since_last_report",
                F.coalesce(F.col("modified_since_last_report") == "Y", F.lit(False)))
)

df.createOrReplaceTempView("v_companies_clean")

### Deduplicate to one row per company (SQL)
Keep the current-name row (`current_name_indicator = 'Y'`) per ACN. From the former-name rows (indicator null) keep the old names (`former_names`), how many there are, and when the current name started (`current_name_start_date`). Apply business rules (status is authoritative over dates).

In [0]:
silver = spark.sql("""
WITH name_history AS (                                   -- former-name rows (indicator null), one summary per company
    SELECT
        acn,
        COUNT_IF(current_name_indicator IS NULL)                                          AS former_name_count,
        ARRAY_SORT(COLLECT_LIST(CASE WHEN current_name_indicator IS NULL THEN company_name END)) AS former_names,
        MAX(CASE WHEN current_name_indicator IS NULL THEN current_name_start_date END)    AS current_name_start_date
    FROM v_companies_clean
    GROUP BY acn
),
current_rows AS (
    SELECT * FROM v_companies_clean WHERE current_name_indicator = 'Y'
)
SELECT
    c.acn,
    c.company_name,
    c.company_type,
    c.company_class,
    c.company_sub_class,
    c.status_code,
    c.registration_date,
    CASE WHEN c.status_code = 'DRGD' THEN c.deregistration_date_reported END AS deregistration_date,
    c.deregistration_date_reported,
    c.previous_state,
    c.state_registration_number,
    c.abn,
    c.abn IS NOT NULL                                        AS has_abn,
    COALESCE(h.former_name_count, 0)                         AS former_name_count,
    h.former_names,
    h.current_name_start_date,
    c.is_acn_named,
    c.status_code = 'DRGD'                                   AS is_deregistered,
    c.status_code IN ('REGD', 'SOFF', 'EXAD', 'NOAC')        AS is_active,
    c.status_code IN ('SOFF', 'EXAD')                        AS is_distressed,
    c.status_code NOT IN ('CNCL', 'DISS')                    AS in_population,
    c.status_code <> 'DRGD' AND c.deregistration_date_reported IS NOT NULL
                                                             AS dereg_date_status_conflict,
    c.company_name_has_export_artifact,
    c.modified_since_last_report,
    c._snapshot_date
FROM current_rows c
LEFT JOIN name_history h ON c.acn = h.acn
""")

### Add code labels
ASIC help-file descriptions for type, class, sub-class, status and previous state; unknown codes give null and are caught by validation.

In [0]:
def label(col_name, mapping):
    m = F.create_map(*[F.lit(x) for kv in mapping.items() for x in kv])
    return F.try_element_at(m, F.col(col_name))   # null if code is unknown (caught in validation)

silver = (silver
    .withColumn("company_type_label",      label("company_type", TYPE_LABELS))
    .withColumn("company_class_label",     label("company_class", CLASS_LABELS))
    .withColumn("company_sub_class_label", label("company_sub_class", SUB_CLASS_LABELS))
    .withColumn("status_label",            label("status_code", STATUS_LABELS))
    .withColumn("previous_state_label",    label("previous_state", PREV_STATE_LABELS))
)

# final column order
silver = silver.select(
    "acn", "company_name",
    "company_type", "company_type_label", "company_class", "company_class_label",
    "company_sub_class", "company_sub_class_label", "status_code", "status_label",
    "registration_date", "deregistration_date", "deregistration_date_reported",
    "previous_state", "previous_state_label", "state_registration_number",
    "abn", "has_abn", "former_name_count", "former_names", "current_name_start_date", "is_acn_named",
    "is_deregistered", "is_active", "is_distressed", "in_population",
    "dereg_date_status_conflict", "company_name_has_export_artifact",
    "modified_since_last_report", "_snapshot_date",
)

### Validation – hard stops
Stops the notebook (and the job) if any rule fails: ACN format and check digit, unknown codes, impossible dates, ABN format/check digit/duplicates, duplicate companies, row-count mismatch.

In [0]:
fails = {}

# --- checks on the cleaned (pre-dedup) rows ---
clean = spark.table("v_companies_clean")
no_y = (clean.groupBy("acn")
    .agg(F.max(F.coalesce((F.col("current_name_indicator") == "Y").cast("int"), F.lit(0))).alias("has_y"),
         F.max(F.when(F.coalesce(F.col("status_code"), F.lit("")) != "CNCL", 1).otherwise(0)).alias("non_cncl"))
    .filter("has_y = 0"))
no_y_acns = no_y.count()
fails["no_current_name_row_non_cncl"] = no_y.filter("non_cncl = 1").count()

pre = clean.agg(
    F.count_if("registration_date_unparseable").alias("reg_unparseable"),
    F.count_if("current_name_start_date_unparseable").alias("name_start_unparseable"),
    F.count_if("deregistration_date_unparseable").alias("dereg_unparseable"),
).first().asDict()
fails.update(pre)

# --- checks on the Silver table ---
acn, abn = F.col("acn"), F.col("abn")
acn_sum = sum(F.substring(acn, i, 1).cast("int") * (9 - i) for i in range(1, 9))
acn_check_ok = ((10 - acn_sum % 10) % 10) == F.substring(acn, 9, 1).cast("int")
abn_sum = sum((F.substring(abn, i, 1).cast("int") - (1 if i == 1 else 0)) * w
              for i, w in enumerate([10, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19], start=1))

post = silver.agg(
    F.count("*").alias("rows"),
    F.countDistinct("acn").alias("distinct_acns"),
    F.count_if(~acn.rlike(r"^[0-9]{9}$") | acn.isNull()).alias("acn_bad_format"),
    F.count_if(acn.rlike(r"^[0-9]{9}$") & ~acn_check_ok).alias("acn_bad_check_digit"),
    F.count_if(F.col("company_type").isNotNull() & F.col("company_type_label").isNull()).alias("unknown_type"),
    F.count_if(F.col("company_class").isNotNull() & F.col("company_class_label").isNull()).alias("unknown_class"),
    F.count_if(F.col("company_sub_class").isNotNull() & F.col("company_sub_class_label").isNull()).alias("unknown_sub_class"),
    F.count_if(F.col("status_code").isNull() | F.col("status_label").isNull()).alias("unknown_status"),
    F.count_if(F.col("previous_state").isNotNull() & F.col("previous_state_label").isNull()).alias("unknown_previous_state"),
    F.count_if(F.col("is_deregistered") & F.col("deregistration_date").isNull()).alias("drgd_without_date"),
    F.count_if(F.col("registration_date") > F.col("_snapshot_date")).alias("reg_after_snapshot"),
    F.count_if(F.col("deregistration_date_reported") > F.col("_snapshot_date")).alias("dereg_after_snapshot"),
    F.count_if(F.col("deregistration_date") < F.col("registration_date")).alias("dereg_before_reg"),
    F.count_if(abn.isNotNull() & ~abn.rlike(r"^[0-9]{11}$")).alias("abn_bad_format"),
    F.count_if(abn.rlike(r"^[0-9]{11}$") & (abn_sum % 89 != 0)).alias("abn_bad_check_digit"),
        F.count_if(F.col("current_name_start_date") > F.col("_snapshot_date")).alias("name_start_after_snapshot"),
    F.count_if(F.size("former_names") != F.col("former_name_count")).alias("former_names_count_mismatch"),
    (F.count(abn) - F.countDistinct(abn)).alias("abn_duplicates"),
).first().asDict()

rows, distinct_acns = post.pop("rows"), post.pop("distinct_acns")
fails.update(post)
fails["duplicate_acns"] = rows - distinct_acns
fails["row_count_mismatch"] = abs(rows - (baseline_acns - no_y_acns))

print(f"Silver rows: {rows:,} | baseline ACNs: {baseline_acns:,} | ACNs without a current-name row: {no_y_acns:,}")
for k, v in fails.items():
    print(f"{'✅' if v == 0 else '❌'} {k}: {v:,}")

# info only (not hard stops)
display(silver.agg(
    F.min("deregistration_date").alias("window_start"), F.max("deregistration_date").alias("window_end"),
    F.count_if("is_deregistered").alias("deregistered"), F.count_if("dereg_date_status_conflict").alias("status_date_conflicts"),
    F.count_if(F.col("registration_date").isNull()).alias("missing_reg_date"), F.count_if("has_abn").alias("with_abn"),
    F.count_if(F.col("former_name_count") > 0).alias("with_former_names"),
    F.count_if((F.col("former_name_count") > 0) & F.col("current_name_start_date").isNull()).alias("former_names_no_start_date"),
    F.count_if(F.col("current_name_start_date") < F.col("registration_date")).alias("name_start_before_registration"),
    F.count_if("modified_since_last_report").alias("modified"), F.count_if("is_acn_named").alias("acn_named")))

bad = {k: v for k, v in fails.items() if v != 0}
assert not bad, f"Silver validation failed: {bad}"

# Write Into Silver Table

In [0]:
(silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE))

print(f"Written {spark.table(SILVER_TABLE).count():,} rows to {SILVER_TABLE} (snapshot {snapshot_date})")

### Table and column descriptions

In [0]:
spark.sql(f"""COMMENT ON TABLE {SILVER_TABLE} IS
'One row per company from the latest ASIC Company Dataset snapshot. Cleaned, typed, deduplicated to the current-name row, with labels and business flags. Rebuilt in full each run.'""")

COLUMN_COMMENTS = {
    "acn": "9-digit ACN (ARBN for registrable bodies). Text to keep leading zeros. Validated incl. check digit.",
    "company_name": "Current registered name. CSV escaping removed, trimmed, repeated spaces collapsed.",
    "company_type": "ASIC company type code (APTY, APUB, FNOS, RACN, CCIV).",
    "company_class": "ASIC liability class code. NONE for foreign and registrable bodies.",
    "company_sub_class": "ASIC sub-class code (e.g. PROP, PSTC super trustee, ULSN not-for-profit).",
    "status_code": "ASIC status code (REGD, DRGD, SOFF, EXAD, NOAC, DISS). Authoritative over dates.",
    "registration_date": "Date registered. 4 missing. May be date joined the ASIC register for some post-1991 transfers.",
    "deregistration_date": "Deregistration date, filled only when status is DRGD. File holds past-year deregistrations only.",
    "deregistration_date_reported": "Deregistration date as reported in the file, regardless of status.",
    "previous_state": "State of registration before the 1991 national scheme. Not current location.",
    "state_registration_number": "Original state registry number. May contain letters.",
    "abn": "ABN as recorded by ASIC. Null when none recorded. Not a check of ABN active status.",
    "has_abn": "True when an ABN is recorded. Indicates trading vs dormant; compare within segment only.",
    "former_name_count": "Number of former-name rows for this ACN.",
    "is_acn_named": "True when the company own ACN appears in its name (numbered company).",
    "former_names": "Previous company names (from former-name rows), sorted alphabetically. Empty list if never renamed.",
    "current_name_start_date": "Date the current name was adopted (from former-name rows). Null if never renamed.",
    "is_deregistered": "Status is DRGD.",
    "is_active": "Status is REGD, SOFF, EXAD or NOAC.",
    "is_distressed": "Status is SOFF (strike-off action) or EXAD (external administration).",
    "in_population": "Included in analysis (excludes CNCL and DISS).",
    "dereg_date_status_conflict": "Deregistration date present but status is not DRGD (likely reinstatement).",
    "company_name_has_export_artifact": "Name contains º (U+00BA), which appears to replace | in the ASIC export.",
    "modified_since_last_report": "Changed since the previous ASIC weekly file. Pipeline use only.",
    "_snapshot_date": "ASIC file snapshot date this row comes from.",
}
for col, text in COLUMN_COMMENTS.items():
    spark.sql(f"ALTER TABLE {SILVER_TABLE} ALTER COLUMN {col} COMMENT '{text}'")
for col in ["company_type", "company_class", "company_sub_class", "status", "previous_state"]:
    name = f"{col}_label"
    spark.sql(f"ALTER TABLE {SILVER_TABLE} ALTER COLUMN {name} COMMENT 'Description of {col.replace('status', 'status_code')} from the ASIC help file'")